In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
from pathlib import Path
import numpy as np
import pandas as pd

# Display versions to verify environment
print(f"pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

# Verify file pathing for your data directory
data_dir = Path("data/processed")
print(f"Data directory status: {'Exists' if data_dir.exists() else 'Needs creation'}")

pandas version: 2.2.2
NumPy version: 2.0.2
Data directory status: Needs creation


In [5]:
from pathlib import Path

# Set the project root path
PROJECT_ROOT = Path("/content/drive/MyDrive/GSSRP_2026/student_performance_project")

# Define the path to the processed data folder
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Create the folder if it does not exist
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Print status to verify setup
print("Project root:", PROJECT_ROOT)
print("Processed-data folder:", PROCESSED_DIR)
print("Folder exists:", PROCESSED_DIR.exists())

Project root: /content/drive/MyDrive/GSSRP_2026/student_performance_project
Processed-data folder: /content/drive/MyDrive/GSSRP_2026/student_performance_project/data/processed
Folder exists: True


In [6]:
import pandas as pd
from pathlib import Path

# 1. Define the correct path
# If your student-mat.csv is in the project root, this will find it:
file_path = "student-mat.csv"

# 2. Safety Check: Verify the file exists before loading
if Path(file_path).exists():
    # Note: Use sep=";" because the CSV files you uploaded use semicolons
    df = pd.read_csv(file_path, sep=";")
    print("Successfully loaded:", file_path)
    print("Current DataFrame shape:", df.shape)
    display(df.head())
else:
    print(f"Error: Could not find '{file_path}'. Check your working directory.")
    # Debug: see where we are currently looking
    print("Current working directory:", Path.cwd())

Successfully loaded: student-mat.csv
Current DataFrame shape: (395, 33)


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,6,10,10


In [7]:
# Identify categorical columns
cat_cols = df.select_dtypes(include="object").columns

# Encode
df_encoded = pd.get_dummies(df, columns=list(cat_cols), drop_first=True)
df_encoded = df_encoded.astype(int)

# Verify
print("New encoded shape:", df_encoded.shape)

New encoded shape: (395, 42)


In [8]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])
print("\nColumn names:")
print(df.columns.tolist())

Number of rows: 395
Number of columns: 33

Column names:
['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2', 'G3']


In [9]:
def inspect_dtypes(dataframe):
    """Creates a readable DataFrame of column types."""
    dtype_table = pd.DataFrame(
        {
            "column": dataframe.columns,
            "dtype": dataframe.dtypes.astype(str).values
        }
    )
    return dtype_table

# Use it to see types before encoding
print("Types before encoding:")
display(inspect_dtypes(df))

Types before encoding:


,column,dtype
0,school,object
1,sex,object
2,age,int64
3,address,object
4,famsize,object
5,Pstatus,object
6,Medu,int64
7,Fedu,int64
8,Mjob,object
9,Fjob,object


In [10]:
# Create a summary table of missing values
missing_summary = (
    df.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)

# Calculate the percentage of missing values per column
missing_summary["missing_percent"] = (
    missing_summary["missing_count"] / len(df) * 100
).round(2)

# Display the top 20 columns with the most missing data
display(missing_summary.head(20))

,missing_count,missing_percent
school,0,0.0
sex,0,0.0
age,0,0.0
address,0,0.0
famsize,0,0.0
Pstatus,0,0.0
Medu,0,0.0
Fedu,0,0.0
Mjob,0,0.0
Fjob,0,0.0


In [11]:
# Check if categorical columns were identified
if len(cat_cols) == 0:
    raise ValueError(
        "No object-type categorical columns were found. "
        "The dataset may already be encoded, or the categorical "
        "columns may use a different data type."
    )

print(f"Categorical columns found: {list(cat_cols)}")
print("Encoding can continue.")

Categorical columns found: ['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']
Encoding can continue.


In [12]:
# Check if categorical columns were identified
if len(cat_cols) == 0:
    raise ValueError(
        "No object-type categorical columns were found. "
        "The dataset may already be encoded, or the categorical "
        "columns may use a different data type."
    )

print(f"Categorical columns found: {list(cat_cols)}")
print("Categorical columns were found. Encoding can continue.")

Categorical columns found: ['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']
Categorical columns were found. Encoding can continue.


In [13]:
# Audit categorical columns before encoding
for column in cat_cols:
    unique_values = df[column].drop_duplicates().tolist()

    print("=" * 70)
    print("Column:", column)
    print("Number of nonmissing categories:", df[column].nunique(dropna=True))
    print("Number of missing values:", df[column].isna().sum())
    print("Example categories:", unique_values[:10])

Column: school
Number of nonmissing categories: 2
Number of missing values: 0
Example categories: ['GP', 'MS']
Column: sex
Number of nonmissing categories: 2
Number of missing values: 0
Example categories: ['F', 'M']
Column: address
Number of nonmissing categories: 2
Number of missing values: 0
Example categories: ['U', 'R']
Column: famsize
Number of nonmissing categories: 2
Number of missing values: 0
Example categories: ['GT3', 'LE3']
Column: Pstatus
Number of nonmissing categories: 2
Number of missing values: 0
Example categories: ['A', 'T']
Column: Mjob
Number of nonmissing categories: 5
Number of missing values: 0
Example categories: ['at_home', 'health', 'other', 'services', 'teacher']
Column: Fjob
Number of nonmissing categories: 5
Number of missing values: 0
Example categories: ['teacher', 'other', 'services', 'health', 'at_home']
Column: reason
Number of nonmissing categories: 4
Number of missing values: 0
Example categories: ['course', 'other', 'home', 'reputation']
Column: g

In [14]:
# Identify columns with high cardinality that might be unique identifiers
possible_identifier_columns = []

for column in cat_cols:
    uniqueness_ratio = df[column].nunique(dropna=False) / len(df)

    # Flag columns where > 90% of values are unique
    if uniqueness_ratio > 0.90:
        possible_identifier_columns.append(
            {
                "column": column,
                "unique_values": df[column].nunique(dropna=False),
                "uniqueness_ratio": round(uniqueness_ratio, 3)
            }
        )

# Review findings
if possible_identifier_columns:
    print("Review these possible identifier columns before encoding:")
    display(pd.DataFrame(possible_identifier_columns))
else:
    print("No likely identifier columns were detected.")

No likely identifier columns were detected.


In [15]:
rows_before = df.shape[0]
columns_before = df.shape[1]
print("Rows before encoding:", rows_before)
print("Columns before encoding:", columns_before)

Rows before encoding: 395
Columns before encoding: 33


In [16]:
# Calculate expected expansion for one-hot encoding
category_count_table = []

for column in cat_cols:
    category_count = df[column].nunique(dropna=True)
    # With drop_first=True, we expect k-1 new columns
    dummy_count = max(category_count - 1, 0)

    category_count_table.append(
        {
            "original_column": column,
            "number_of_categories": category_count,
            "expected_dummy_columns": dummy_count
        }
    )

# Convert to DataFrame for a clean view
category_count_df = pd.DataFrame(category_count_table)
display(category_count_df)

# Total expected columns check
print(f"Total dummy columns to be added: {category_count_df['expected_dummy_columns'].sum()}")

,original_column,number_of_categories,expected_dummy_columns
0,school,2,1
1,sex,2,1
2,address,2,1
3,famsize,2,1
4,Pstatus,2,1
5,Mjob,5,4
6,Fjob,5,4
7,reason,4,3
8,guardian,3,2
9,schoolsup,2,1


Total dummy columns to be added: 26


In [17]:
# Calculate total expected columns
number_of_numeric_columns = df.shape[1] - len(cat_cols)
expected_dummy_columns = category_count_df["expected_dummy_columns"].sum()
expected_total_columns = number_of_numeric_columns + expected_dummy_columns

print("Original numeric columns:", number_of_numeric_columns)
print("Expected dummy columns:", expected_dummy_columns)
print("Expected final column count:", expected_total_columns)

Original numeric columns: 16
Expected dummy columns: 26
Expected final column count: 42


In [18]:
# One-hot encode nominal categorical columns
cat_cols = df.select_dtypes(include="object").columns

# Apply one-hot encoding with drop_first=True to avoid the dummy-variable trap
df_encoded = pd.get_dummies(df, columns=list(cat_cols), drop_first=True)

# Print the resulting shape
print("Encoded shape:", df_encoded.shape)

Encoded shape: (395, 42)


In [19]:
# Convert categorical features into binary/dummy variables
df_encoded = pd.get_dummies(
    df,
    columns=list(cat_cols),
    drop_first=True
)

# Optional: Convert True/False to 1/0 for cleaner numerical representation
df_encoded = df_encoded.astype(int)

# Verify the result
print("New DataFrame shape:", df_encoded.shape)

New DataFrame shape: (395, 42)


In [20]:
rows_after = df_encoded.shape[0]
columns_after = df_encoded.shape[1]
print("Rows before encoding:", rows_before)
print("Rows after encoding:", rows_after)
print("\nColumns before encoding:", columns_before)
print("Columns after encoding:", columns_after)
print("\nNet change in columns:", columns_after - columns_before)

Rows before encoding: 395
Rows after encoding: 395

Columns before encoding: 33
Columns after encoding: 42

Net change in columns: 9


In [21]:
# Assuming columns_after was calculated using: columns_after = df_encoded.shape[1]

print("Expected final column count:", expected_total_columns)
print("Actual final column count:", columns_after)

if columns_after == expected_total_columns:
    print("Column-count check passed.")
else:
    print(
        "Column-count check requires review. "
        "Missing values or unusual category structures may affect the result."
    )

Expected final column count: 42
Actual final column count: 42
Column-count check passed.


In [22]:
# Identify columns removed and added during encoding
original_columns = set(df.columns)
encoded_columns = set(df_encoded.columns)

# Find columns that were transformed (removed from the original set)
removed_columns = [
    column for column in df.columns
    if column not in encoded_columns
]

# Find the new binary dummy columns created
new_columns = [
    column for column in df_encoded.columns
    if column not in original_columns
]

print("Original categorical columns removed:")
print(removed_columns)
print("\nNumber of new dummy columns:", len(new_columns))
print("\nFirst 30 new dummy columns:")
print(new_columns[:30])

Original categorical columns removed:
['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']

Number of new dummy columns: 26

First 30 new dummy columns:
['school_MS', 'sex_M', 'address_U', 'famsize_LE3', 'Pstatus_T', 'Mjob_health', 'Mjob_other', 'Mjob_services', 'Mjob_teacher', 'Fjob_health', 'Fjob_other', 'Fjob_services', 'Fjob_teacher', 'reason_home', 'reason_other', 'reason_reputation', 'guardian_mother', 'guardian_other', 'schoolsup_yes', 'famsup_yes', 'paid_yes', 'activities_yes', 'nursery_yes', 'higher_yes', 'internet_yes', 'romantic_yes']


In [23]:
display(df_encoded.head())

,age,Medu,Fedu,traveltime,studytime,failures,famrel,freetime,goout,Dalc,...,guardian_mother,guardian_other,schoolsup_yes,famsup_yes,paid_yes,activities_yes,nursery_yes,higher_yes,internet_yes,romantic_yes
0,18,4,4,2,2,0,4,3,4,1,...,1,0,1,0,0,0,1,1,0,0
1,17,1,1,1,2,0,5,3,3,1,...,0,0,0,1,0,0,0,1,1,0
2,15,1,1,1,2,3,4,3,2,2,...,1,0,1,0,1,0,1,1,1,0
3,15,4,2,1,3,0,3,2,2,1,...,1,0,0,1,1,1,1,1,1,1
4,16,3,3,1,2,0,4,3,2,1,...,0,0,0,1,1,0,1,1,0,0


In [24]:
display(df_encoded[new_columns].head())

,school_MS,sex_M,address_U,famsize_LE3,Pstatus_T,Mjob_health,Mjob_other,Mjob_services,Mjob_teacher,Fjob_health,...,guardian_mother,guardian_other,schoolsup_yes,famsup_yes,paid_yes,activities_yes,nursery_yes,higher_yes,internet_yes,romantic_yes
0,0,0,1,0,0,0,0,0,0,0,...,1,0,1,0,0,0,1,1,0,0
1,0,0,1,0,1,0,0,0,0,0,...,0,0,0,1,0,0,0,1,1,0
2,0,0,1,1,1,0,0,0,0,0,...,1,0,1,0,1,0,1,1,1,0
3,0,0,1,0,1,1,0,0,0,0,...,1,0,0,1,1,1,1,1,1,1
4,0,0,1,0,1,0,1,0,0,0,...,0,0,0,1,1,0,1,1,0,0


In [25]:
boolean_columns = df_encoded.select_dtypes(include="bool").columns

In [26]:
print("Number of Boolean dummy columns:", len(boolean_columns))

Number of Boolean dummy columns: 0


In [27]:
# Identify which columns were created by get_dummies (if you haven't already)
# Alternatively, you can convert all columns that only contain 0s and 1s
boolean_columns = [
    col for col in df_encoded.columns
    if df_encoded[col].isin([0, 1]).all()
]

# Convert these columns to the most memory-efficient integer type
df_encoded[boolean_columns] = df_encoded[boolean_columns].astype("int8")

print("Boolean dummy variables converted to 0 and 1 (int8 type).")
print(f"Memory usage optimized for {len(boolean_columns)} columns.")

Boolean dummy variables converted to 0 and 1 (int8 type).
Memory usage optimized for 26 columns.


In [28]:
display(df_encoded.head())

,age,Medu,Fedu,traveltime,studytime,failures,famrel,freetime,goout,Dalc,...,guardian_mother,guardian_other,schoolsup_yes,famsup_yes,paid_yes,activities_yes,nursery_yes,higher_yes,internet_yes,romantic_yes
0,18,4,4,2,2,0,4,3,4,1,...,1,0,1,0,0,0,1,1,0,0
1,17,1,1,1,2,0,5,3,3,1,...,0,0,0,1,0,0,0,1,1,0
2,15,1,1,1,2,3,4,3,2,2,...,1,0,1,0,1,0,1,1,1,0
3,15,4,2,1,3,0,3,2,2,1,...,1,0,0,1,1,1,1,1,1,1
4,16,3,3,1,2,0,4,3,2,1,...,0,0,0,1,1,0,1,1,0,0


In [29]:
# Identify any remaining columns with 'object' data type
remaining_object_columns = (
    df_encoded
    .select_dtypes(include="object")
    .columns
    .tolist()
)

# Audit the result
if len(remaining_object_columns) == 0:
    print("Success: No object-type columns remain. The dataset is fully numeric.")
else:
    print("Warning: The following object-type columns still exist and may require further manual encoding:")
    print(remaining_object_columns)

Success: No object-type columns remain. The dataset is fully numeric.


In [30]:
# Final validation: Ensure no object-type columns exist
assert len(remaining_object_columns) == 0, (
    f"Some object-type columns remain after encoding: {remaining_object_columns}"
)

print("Object-column check passed. Dataset is ready for modeling.")

Object-column check passed. Dataset is ready for modeling.


In [31]:
# Final validation: Ensure the number of rows remains constant
assert df_encoded.shape[0] == df.shape[0], (
    f"The number of rows changed: {df.shape[0]} original vs {df_encoded.shape[0]} encoded."
)

print("Row-count check passed. Integrity of records maintained.")

Row-count check passed. Integrity of records maintained.


In [32]:
# Final validation: Ensure the index order and labels were preserved
assert df_encoded.index.equals(df.index), (
    "The row index changed or was reordered during encoding."
)

print("Row-index check passed. Data alignment is preserved.")

Row-index check passed. Data alignment is preserved.


In [33]:
# Validate that the target variable (G3) is preserved and unmodified
if "G3" in df.columns:
    assert "G3" in df_encoded.columns, (
        "The G3 target variable is missing after encoding."
    )

    # Ensure the values and order of the target are identical to the original
    pd.testing.assert_series_equal(
        df_encoded["G3"],
        df["G3"],
        check_names=True
    )
    print("G3 target-variable check passed.")
else:
    print("G3 was not found in the current dataset.")

G3 target-variable check passed.


In [34]:
# Check for any duplicate column names in the encoded DataFrame
duplicate_column_count = df_encoded.columns.duplicated().sum()

print("Duplicate column names detected:", duplicate_column_count)

# Assert that there are no duplicates to ensure clean input for modeling
assert duplicate_column_count == 0, (
    "Duplicate column names were detected. Please review your input columns."
)

print("Duplicate-column check passed. The structure is clean.")

Duplicate column names detected: 0
Duplicate-column check passed. The structure is clean.


In [35]:
missing_after = df_encoded.isna().sum().sum()
print("Total missing values before encoding:", df.isna().sum().sum())
print("Total missing values after encoding:", missing_after)

Total missing values before encoding: 0
Total missing values after encoding: 0


In [36]:
# Final check for any remaining missing values
columns_with_missing_values = (
    df_encoded.isna()
    .sum()
    .loc[lambda values: values > 0]
    .sort_values(ascending=False)
)

if columns_with_missing_values.empty:
    print("No missing values remain. The dataset is clean.")
else:
    print("Warning: Columns still containing missing values:")
    display(columns_with_missing_values.to_frame("missing_count"))

No missing values remain. The dataset is clean.


In [37]:
# Consolidated sanity checks for the encoded dataset
assert df_encoded.shape[0] == df.shape[0], (
    "Sanity check failed: row count changed."
)
assert df_encoded.index.equals(df.index), (
    "Sanity check failed: row index changed."
)
assert not df_encoded.columns.duplicated().any(), (
    "Sanity check failed: duplicate column names exist."
)

# Verify all columns are now numeric
remaining_objects = df_encoded.select_dtypes(include="object").columns.tolist()
assert len(remaining_objects) == 0, (
    f"Sanity check failed: object columns remain: {remaining_objects}"
)

# Validate the target variable integrity
if "G3" in df.columns:
    assert "G3" in df_encoded.columns, (
        "Sanity check failed: G3 is missing."
    )
    # Ensure values, index, and data type are identical
    pd.testing.assert_series_equal(df_encoded["G3"], df["G3"])

print("All one-hot encoding sanity checks passed. The dataset is fully validated.")

All one-hot encoding sanity checks passed. The dataset is fully validated.


In [38]:
# Create a comprehensive summary report of the encoding process
encoding_summary = pd.DataFrame(
    {
        "measurement": [
            "Rows before encoding",
            "Rows after encoding",
            "Columns before encoding",
            "Columns after encoding",
            "Categorical columns encoded",
            "New dummy columns",
            "Remaining object columns",
            "Missing values after encoding"
        ],
        "value": [
            df.shape[0],
            df_encoded.shape[0],
            df.shape[1],
            df_encoded.shape[1],
            len(cat_cols),
            len(new_columns),
            len(remaining_object_columns),
            df_encoded.isna().sum().sum()
        ]
    }
)

# Display the summary for a final review
display(encoding_summary)

,measurement,value
0,Rows before encoding,395
1,Rows after encoding,395
2,Columns before encoding,33
3,Columns after encoding,42
4,Categorical columns encoded,17
5,New dummy columns,26
6,Remaining object columns,0
7,Missing values after encoding,0


In [40]:
from google.colab import drive
from pathlib import Path
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Mount Google Drive
# ------------------------------------------------------------
drive.mount("/content/drive")

# ------------------------------------------------------------
# 2. Define project paths
# ------------------------------------------------------------
PROJECT_ROOT = Path("/content/drive/MyDrive/GSSRP_2026/student_performance_project")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

INPUT_FILE = PROCESSED_DIR / "student_data_clean.csv"
OUTPUT_FILE = PROCESSED_DIR / "student_data_encoded.csv"

# ------------------------------------------------------------
# 3. Load the cleaned dataset
# ------------------------------------------------------------
if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Input file not found: {INPUT_FILE}")

df = pd.read_csv(INPUT_FILE, sep=None, engine="python")
print(f"Dataset loaded: {INPUT_FILE.name}")
print(f"Original shape: {df.shape}")

# ------------------------------------------------------------
# 4. Identify categorical columns
# ------------------------------------------------------------
cat_cols = df.select_dtypes(include="object").columns
print(f"\nCategorical columns ({len(cat_cols)}): {list(cat_cols)}")

# ------------------------------------------------------------
# 5. One-hot encode categorical columns
# ------------------------------------------------------------
# Using drop_first=True to avoid the dummy-variable trap
df_encoded = pd.get_dummies(df, columns=list(cat_cols), drop_first=True)

# ------------------------------------------------------------
# 6. Convert Boolean dummy variables to integers
# ------------------------------------------------------------
boolean_columns = df_encoded.select_dtypes(include="bool").columns
df_encoded[boolean_columns] = df_encoded[boolean_columns].astype("int8")

print(f"Encoded shape: {df_encoded.shape}")

# ------------------------------------------------------------
# 7. Final Sanity Checks
# ------------------------------------------------------------
assert df_encoded.shape[0] == df.shape[0], "Row count mismatch!"
assert df_encoded.index.equals(df.index), "Row index mismatch!"
assert not df_encoded.columns.duplicated().any(), "Duplicate columns detected!"
assert len(df_encoded.select_dtypes(include="object").columns) == 0, "Object columns remain!"

if "G3" in df.columns:
    assert "G3" in df_encoded.columns, "Target variable G3 missing!"
    pd.testing.assert_series_equal(df_encoded["G3"], df["G3"])

print("All one-hot encoding sanity checks passed.")

# ------------------------------------------------------------
# 8. Save and Verify
# ------------------------------------------------------------
df_encoded.to_csv(OUTPUT_FILE, index=False)
print(f"Saved: {OUTPUT_FILE.name}")

# Reload to verify
df_encoded_check = pd.read_csv(OUTPUT_FILE)
assert df_encoded_check.shape == df_encoded.shape, "Reload verification failed!"
print("Saved-file verification passed.")

# Preview the results
display(df_encoded.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


FileNotFoundError: Input file not found: /content/drive/MyDrive/GSSRP_2026/student_performance_project/data/processed/student_data_clean.csv

In [41]:
# Update the INPUT_FILE path here
INPUT_FILE = PROCESSED_DIR / "student-mat.csv"
# If the file is in a raw folder
INPUT_FILE = PROJECT_ROOT / "data" / "raw" / "student-mat.csv"
# Debugging check: Run this if you are still getting an error
import os
print("Does file exist?", INPUT_FILE.exists())
print("Looking in path:", INPUT_FILE)

Does file exist? False
Looking in path: /content/drive/MyDrive/GSSRP_2026/student_performance_project/data/raw/student-mat.csv


In [42]:
# --- Update this section in your script ---
# If the file is named student-mat.csv, use this:
INPUT_FILE = PROCESSED_DIR / "student-mat.csv"

# --- OR, if you have already saved it as student_data_clean.csv ---
# Ensure the file is actually in the 'processed' folder:
# INPUT_FILE = PROCESSED_DIR / "student_data_clean.csv"

In [43]:
import os

# Check if the path exists
print(f"Checking path: {INPUT_FILE}")
if INPUT_FILE.exists():
    print("✅ File found! Proceeding to load.")
else:
    print("❌ File NOT found. Please check your folder structure.")
    # This lists the files in your processed directory so you can see the correct name
    print("Files in directory:", list(PROCESSED_DIR.iterdir()))

Checking path: /content/drive/MyDrive/GSSRP_2026/student_performance_project/data/processed/student-mat.csv
❌ File NOT found. Please check your folder structure.
Files in directory: [PosixPath('/content/drive/MyDrive/GSSRP_2026/student_performance_project/data/processed/X_full.csv'), PosixPath('/content/drive/MyDrive/GSSRP_2026/student_performance_project/data/processed/y_full.csv'), PosixPath('/content/drive/MyDrive/GSSRP_2026/student_performance_project/data/processed/full_data.csv'), PosixPath('/content/drive/MyDrive/GSSRP_2026/student_performance_project/data/processed/y_full.parquet'), PosixPath('/content/drive/MyDrive/GSSRP_2026/student_performance_project/data/processed/X_full.parquet'), PosixPath('/content/drive/MyDrive/GSSRP_2026/student_performance_project/data/processed/full_data.parquet')]


In [45]:
# Change this line to point to your raw data location
# Example: If the file is in the root of your GSSRP folder
INPUT_FILE = Path("/content/drive/MyDrive/GSSRP/student-mat.csv")

In [46]:
# ------------------------------------------------------------
# 3. Load the cleaned dataset (Fixed Version)
# ------------------------------------------------------------
import os

# Define the target file name
target_filename = "student-mat.csv"
INPUT_FILE = PROCESSED_DIR / target_filename

# Debugging: List files if it's not found
if not INPUT_FILE.exists():
    available_files = [f.name for f in PROCESSED_DIR.iterdir()]
    raise FileNotFoundError(
        f"Could not find '{target_filename}' in {PROCESSED_DIR}.\n"
        f"Available files are: {available_files}\n"
        "Please check if the file name is different."
    )

df = pd.read_csv(INPUT_FILE, sep=None, engine="python")
print(f"✅ Dataset loaded successfully: {INPUT_FILE.name}")
print(f"Original shape: {df.shape}")

FileNotFoundError: Could not find 'student-mat.csv' in /content/drive/MyDrive/GSSRP_2026/student_performance_project/data/processed.
Available files are: ['X_full.csv', 'y_full.csv', 'full_data.csv', 'y_full.parquet', 'X_full.parquet', 'full_data.parquet']
Please check if the file name is different.

In [47]:
# Search for all CSV files in your project directory
found_files = list(PROJECT_ROOT.rglob("*.csv"))
print("Found these CSV files in your project:")
for f in found_files:
    print(f)

Found these CSV files in your project:
/content/drive/MyDrive/GSSRP_2026/student_performance_project/data/processed/X_full.csv
/content/drive/MyDrive/GSSRP_2026/student_performance_project/data/processed/y_full.csv
/content/drive/MyDrive/GSSRP_2026/student_performance_project/data/processed/full_data.csv


In [48]:
# Change this line based on what the search found above
INPUT_FILE = PROJECT_ROOT / "data" / "raw" / "student-mat.csv"

In [49]:
!find /content/drive/MyDrive/GSSRP_2026/student_performance_project -name "*.csv"

/content/drive/MyDrive/GSSRP_2026/student_performance_project/data/processed/X_full.csv
/content/drive/MyDrive/GSSRP_2026/student_performance_project/data/processed/y_full.csv
/content/drive/MyDrive/GSSRP_2026/student_performance_project/data/processed/full_data.csv


In [50]:
import os
for root, dirs, files in os.walk("/content/drive/MyDrive/GSSRP_2026"):
    for file in files:
        if "student" in file and file.endswith(".csv"):
            print(os.path.join(root, file))

In [51]:
from pathlib import Path
import pandas as pd

# Use this path AFTER you have dragged and dropped the file into the sidebar
INPUT_FILE = Path("/content/student-mat.csv")

# Load Dataset
df = pd.read_csv(INPUT_FILE, sep=None, engine="python")
print(f"✅ Loaded: {df.shape}")

# Proceed with your logic
cat_cols = df.select_dtypes(include="object").columns
df_encoded = pd.get_dummies(df, columns=list(cat_cols), drop_first=True)
bool_cols = df_encoded.select_dtypes(include="bool").columns
df_encoded[bool_cols] = df_encoded[bool_cols].astype("int8")

# Display first few rows to confirm success
display(df_encoded.head())

✅ Loaded: (395, 33)


,age,Medu,Fedu,traveltime,studytime,failures,famrel,freetime,goout,Dalc,...,guardian_mother,guardian_other,schoolsup_yes,famsup_yes,paid_yes,activities_yes,nursery_yes,higher_yes,internet_yes,romantic_yes
0,18,4,4,2,2,0,4,3,4,1,...,1,0,1,0,0,0,1,1,0,0
1,17,1,1,1,2,0,5,3,3,1,...,0,0,0,1,0,0,0,1,1,0
2,15,1,1,1,2,3,4,3,2,2,...,1,0,1,0,1,0,1,1,1,0
3,15,4,2,1,3,0,3,2,2,1,...,1,0,0,1,1,1,1,1,1,1
4,16,3,3,1,2,0,4,3,2,1,...,0,0,0,1,1,0,1,1,0,0


In [52]:
# 1. Define X_full (features) and y_full (target)
# We drop G3 from the features because it is our target variable (preventing leakage)
X_full = df_encoded.drop(columns=["G3"])
y_full = df_encoded["G3"]

print(f"Full-information features shape: {X_full.shape}")
print(f"Target variable shape: {y_full.shape}")

# 2. Validation Checks
assert "G3" not in X_full.columns, "Error: G3 still in features!"
assert "G1" in X_full.columns and "G2" in X_full.columns, "Error: G1/G2 missing!"
assert X_full.shape[0] == y_full.shape[0], "Error: Row mismatch!"

# 3. Install necessary library for Parquet
!pip install pyarrow

# 4. Save the files
# Saving as both Parquet (efficient) and CSV (readable)
X_full.to_parquet(PROCESSED_DIR / "X_full.parquet")
y_full.to_frame().to_parquet(PROCESSED_DIR / "y_full.parquet")
df_encoded.to_parquet(PROCESSED_DIR / "full_data.parquet")

X_full.to_csv(PROCESSED_DIR / "X_full.csv", index=False)
y_full.to_csv(PROCESSED_DIR / "y_full.csv", index=False)
df_encoded.to_csv(PROCESSED_DIR / "full_data.csv", index=False)

print("\n✅ Session 20: Files saved to 'data/processed/'")

Full-information features shape: (395, 41)
Target variable shape: (395,)

✅ Session 20: Files saved to 'data/processed/'


In [53]:
X_full = df_encoded.drop(columns=["G3"])
y = df_encoded["G3"]
print("Full-information features:", X_full.shape[1])

Full-information features: 41


# GSSRP 2026 — Session 20
## Feature Engineering II: Full-Information Feature Set
This notebook creates the full-information modeling dataset. The predictors include
G1 and G2, while G3 is separated as the target variable.

In [54]:
import pandas as pd
print("Pandas version:", pd.__version__)

Pandas version: 2.2.2


In [55]:
import pandas as pd
from pathlib import Path

# 1. Ensure df_encoded is ready
if "df_encoded" not in globals():
    # Adjust path if needed; use the one we established in Session 19
    INPUT_FILE = Path("/content/student_data_encoded.csv")
    df_encoded = pd.read_csv(INPUT_FILE)
    print("✅ Loaded df_encoded from file.")
else:
    print("✅ df_encoded is already in memory.")

# 2. Assemble Full-Information Sets
X_full = df_encoded.drop(columns=["G3"])
y_full = df_encoded["G3"]

print(f"Full-information features shape: {X_full.shape}")
print(f"Target variable shape: {y_full.shape}")

# 3. Required Validation (Per Session 20 Checklist)
assert "G3" not in X_full.columns, "❌ Error: G3 still in features!"
assert "G1" in X_full.columns and "G2" in X_full.columns, "✅ G1 and G2 present."
assert X_full.shape[0] == y_full.shape[0], "❌ Error: Row mismatch!"

# 4. Save to processed directory (as per requirements)
PROCESSED_DIR = Path("/content/drive/MyDrive/GSSRP_2026/student_performance_project/data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

X_full.to_parquet(PROCESSED_DIR / "X_full.parquet")
y_full.to_frame().to_parquet(PROCESSED_DIR / "y_full.parquet")
df_encoded.to_parquet(PROCESSED_DIR / "full_data.parquet")

print("\n✅ Session 20: Full-Information dataset assembled and saved.")

✅ df_encoded is already in memory.
Full-information features shape: (395, 41)
Target variable shape: (395,)

✅ Session 20: Full-Information dataset assembled and saved.


In [56]:
import pandas as pd

# ------------------------------------------------------------
# 1. Validate Input DataFrame
# ------------------------------------------------------------
if "df_encoded" not in globals():
    raise NameError("df_encoded is not defined. Load your dataset first.")

required_columns = ["G1", "G2", "G3"]
missing = [c for c in required_columns if c not in df_encoded.columns]
if missing:
    raise KeyError(f"Required columns missing: {', '.join(missing)}")

# ------------------------------------------------------------
# 2. Assemble Full-Information Features and Target
# ------------------------------------------------------------
X_full = df_encoded.drop(columns=["G3"]).copy()
y = df_encoded["G3"].copy()

# ------------------------------------------------------------
# 3. Validation Logic (Checklist Compliance)
# ------------------------------------------------------------
assert "G3" not in X_full.columns, "Target leakage detected: G3 in X_full."
assert "G1" in X_full.columns and "G2" in X_full.columns, "G1/G2 missing from X_full."
assert len(X_full) == len(y), "Row count mismatch between X_full and y."
assert X_full.index.equals(y.index), "Index mismatch between X_full and y."

# ------------------------------------------------------------
# 4. Results Summary
# ------------------------------------------------------------
print("=" * 60)
print("SESSION 20: FULL-INFORMATION DATASET SUMMARY")
print("=" * 60)
print(f"Features shape: {X_full.shape}")
print(f"Target shape: {y.shape}")
print(f"G1 & G2 Included: {'G1' in X_full.columns and 'G2' in X_full.columns}")
print(f"G3 Excluded from X: {'G3' not in X_full.columns}")
print(f"Missing Values: {X_full.isna().sum().sum() + y.isna().sum()}")
print("=" * 60)
print("✅ Verification passed: Full-information dataset is ready.")

SESSION 20: FULL-INFORMATION DATASET SUMMARY
Features shape: (395, 41)
Target shape: (395,)
G1 & G2 Included: True
G3 Excluded from X: True
Missing Values: 0
✅ Verification passed: Full-information dataset is ready.


In [57]:
# 1. Validation: Target Exclusion
assert "G3" not in X_full.columns, "❌ Target leakage detected: G3 is still present in X_full."
print("✅ Verification passed: G3 is correctly excluded from the feature matrix.")

# 2. Validation: Inclusion of Required Predictors
assert "G1" in X_full.columns and "G2" in X_full.columns, "❌ G1 or G2 missing from X_full."
print("✅ Verification passed: G1 and G2 are present in the feature matrix.")

# 3. Validation: Dimensional Integrity
assert len(X_full) == len(y), "❌ Row count mismatch between features and target."
print("✅ Verification passed: Row counts match.")

✅ Verification passed: G3 is correctly excluded from the feature matrix.
✅ Verification passed: G1 and G2 are present in the feature matrix.
✅ Verification passed: Row counts match.


### Summary

The full-information feature set ($X_{full}$) has been successfully decoupled from the target variable ($y$). The feature matrix now contains all necessary predictor variables, including $G1$ and $G2$, while the target vector ($y$) exclusively holds the $G3$ final grades. All integrity checks—including row count matching, index alignment, and the exclusion of the target—have passed.

### Interpretation

* **Target Exclusion:** The removal of `G3` from the feature matrix is verified. This ensures no data leakage occurs, preventing the model from having "advance knowledge" of the outcome during the training phase.
* **Feature Count:** Your feature matrix now contains exactly one fewer column than the original encoded dataset ($df_{encoded.shape[1]} - 1$). This confirms that the separation was isolated strictly to the target column without any loss or duplication of predictor variables.

### Recommendation

For the storage of this dataset, I recommend using the **Apache Parquet** format.

* **Why Parquet?** Unlike CSV, Parquet is a columnar storage format that preserves your specific data types (such as `int8` for your encoded booleans) and is significantly more efficient for both disk space and I/O performance.
* **Workflow:** Save your artifacts using `X_full.to_parquet("X_full.parquet")` and `y.to_frame().to_parquet("y_full.parquet")` within your `data/processed/` directory. If you require human-readable backups for inspection, you may simultaneously save them as `.csv` files, but use the Parquet versions as your primary source for the modeling sessions to follow.

## Session 20 Prompt-Engineered Explanation Task

### Summary

The full-information feature set is **correctly separated** from the target variable. The verification results confirm that the target ($G3$) has been successfully excluded from the feature matrix ($X_{full}$), while the critical prior-grade predictors ($G1$ and $G2$) remain intact, meeting all requirements for a valid full-information modeling scenario.

### Interpretation

* **Target-Feature Separation:** By dropping the column `G3` from the feature matrix, you have effectively isolated the dependent variable. This is critical because including `G3` in the features would lead to **target leakage**—a condition where the model gains "foreknowledge" of the final result during training, leading to artificially inflated performance metrics and a failure to generalize to new, unseen data.
* **Feature Set Composition:** Your feature matrix $X_{full}$ now contains **32** features (or the total count post-encoding minus 1). The presence of $G1$ and $G2$ confirms that this is a "full-information" dataset, which allows the model to leverage a student’s academic history throughout the semester to predict the final outcome.

### Recommendation

I recommend saving your datasets in the **Apache Parquet** format.

Parquet is highly optimized for machine learning workflows because it stores data columnarly, which preserves the specific data types (such as `int8`) assigned during your feature engineering process. This ensures that when you reload your data for modeling in the next session, the memory footprint remains low and the schema is perfectly maintained. You may save the files using the `.to_parquet()` method:

```python
X_full.to_parquet(PROCESSED_DIR / "X_full.parquet")
y.to_frame().to_parquet(PROCESSED_DIR / "y_full.parquet")

```

If a human-readable format is required for external audits, you can also save them as `.csv` files, but use the Parquet files as the primary source for your model training.


In [58]:
# 1. Status Check
print("X_full exists:", "X_full" in globals())
print("y exists:", "y" in globals())

# 2. Generate facts for the Prompt
# Note: Ensure X_full and y are defined from previous cells before running this
feature_count = X_full.shape[1]
row_count = X_full.shape[0]
target_name = y.name
target_excluded = "G3" not in X_full.columns
g1_included = "G1" in X_full.columns
g2_included = "G2" in X_full.columns

# 3. Print Results for Documentation
print("-" * 30)
print("FACTS FOR PROMPT:")
print(f"Number of rows: {row_count}")
print(f"Number of features: {feature_count}")
print(f"Target variable: {target_name}")
print(f"G3 excluded from X_full: {target_excluded}")
print(f"G1 included in X_full: {g1_included}")
print(f"G2 included in X_full: {g2_included}")
print("-" * 30)

# 4. Final Validation Assertion
assert target_excluded == True, "❌ Target leakage detected: G3 is still present!"
assert g1_included == True and g2_included == True, "❌ G1 or G2 is missing!"
print("✅ Verification passed: Full-information dataset is prepared.")

X_full exists: True
y exists: True
------------------------------
FACTS FOR PROMPT:
Number of rows: 395
Number of features: 41
Target variable: G3
G3 excluded from X_full: True
G1 included in X_full: True
G2 included in X_full: True
------------------------------
✅ Verification passed: Full-information dataset is prepared.


Based on the GSSRP 2026 Session 20 curriculum, you are currently in the **Feature Engineering II** phase. To determine if you have completed the required steps for the "full-information feature set," check if your Google Colab environment has the following:

### 1. Verification of Objects

Ensure the following objects exist in your workspace (as per Section 3):

*
**`X_full`**: Contains your predictor variables, including $G1$ and $G2$, with the target $G3$ successfully dropped.


*
**`y`**: Contains only the target variable, $G3$.



### 2. Required Validation Steps (Section 4)

You should have already run the verification cells to confirm the following criteria are met :

*
**Target Exclusion:** The target variable $G3$ is removed from `X_full`.


*
**Predictor Inclusion:** $G1$ and $G2$ are present in `X_full`.


*
**Row Alignment:** The row counts for `X_full` and `y` match.


*
**Feature Count:** You have noted the exact feature count from `X_full.shape[1]`.



### 3. Prompt-Engineering Documentation

You must have a Text cell in your notebook labeled **"Session 20 Prompt-Engineered Explanation Task"** that contains:

1.
**Prompt Used:** The template populated with your specific feature and row counts.


2.
**AI-Generated Response:** The output you received from the AI model verifying your dataset.


3.
**Student Verification:** A statement confirming you verified the AI's response against your Python output .



### 4. GitHub Deliverable (Section 8)

Note: This is a separate step done in VS Code, not Colab.

* You must have saved your dataset as **Parquet files** (`X_full.parquet`, `y_full.parquet`, `full_information_dataset.parquet`) in your `data/processed/` folder.


* These files should be committed and pushed to your GitHub repository.



**If you have performed the Python validations in your notebook, have the prompt/response recorded, and have committed the Parquet files to GitHub, you have completed the session requirements.** If you are missing any of these, please follow the specific sections in your session materials to finalize your deliverable.

In [59]:
feature_count = X_full.shape[1]
row_count = X_full.shape[0]
target_excluded = "G3" not in X_full.columns
g1_included = "G1" in X_full.columns
g2_included = "G2" in X_full.columns

print("Number of rows:", row_count)
print("Number of features:", feature_count)
print("Target variable: G3")
print("G3 excluded from X_full:", target_excluded)
print("G1 included in X_full:", g1_included)
print("G2 included in X_full:", g2_included)

Number of rows: 395
Number of features: 41
Target variable: G3
G3 excluded from X_full: True
G1 included in X_full: True
G2 included in X_full: True


In [60]:
# Assemble Features and Target
X_full = df_encoded.drop(columns=["G3"]).copy()
y = df_encoded["G3"].copy()

# Validation Checks
assert "G3" not in X_full.columns, "Error: G3 present in features!"
assert "G1" in X_full.columns and "G2" in X_full.columns, "Error: G1/G2 missing!"
assert len(X_full) == len(y), "Error: Row mismatch!"

# Print metadata for the Prompt
print(f"Number of rows: {X_full.shape[0]}")
print(f"Number of features: {X_full.shape[1]}")
print(f"Target variable: {y.name}")
print(f"G3 excluded from X_full: {'G3' not in X_full.columns}")
print(f"G1 included in X_full: {'G1' in X_full.columns}")
print(f"G2 included in X_full: {'G2' in X_full.columns}")

Number of rows: 395
Number of features: 41
Target variable: G3
G3 excluded from X_full: True
G1 included in X_full: True
G2 included in X_full: True


In [61]:
from pathlib import Path
PROCESSED_DIR = Path("/content/drive/MyDrive/GSSRP/data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Save as Parquet (Required format for session 20)
X_full.to_parquet(PROCESSED_DIR / "X_full.parquet")
y.to_frame().to_parquet(PROCESSED_DIR / "y_full.parquet")
df_encoded.to_parquet(PROCESSED_DIR / "full_data.parquet")

print("✅ Parquet files saved to data/processed/")

✅ Parquet files saved to data/processed/


In [62]:
import pandas as pd
from pathlib import Path

# 1. Ensure directories exist
PROCESSED_DIR = Path("/content/drive/MyDrive/GSSRP/data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# 2. Assemble Full-Information Features and Target
# X_full includes G1 and G2; y is exclusively G3
X_full = df_encoded.drop(columns=["G3"]).copy()
y = df_encoded["G3"].copy()

# 3. Required Validation (Per Section 8 Checklist)
assert "G3" not in X_full.columns, "❌ Target leakage detected: G3 is still in features."
assert "G1" in X_full.columns and "G2" in X_full.columns, "❌ G1 or G2 missing."
assert len(X_full) == len(y), "❌ Row count mismatch."

# 4. Save to Parquet (The required format)
X_full.to_parquet(PROCESSED_DIR / "X_full.parquet")
y.to_frame().to_parquet(PROCESSED_DIR / "y_full.parquet")
df_encoded.to_parquet(PROCESSED_DIR / "full_data.parquet")

# 5. Final Output Summary
print("=" * 60)
print("SESSION 20: FULL-INFORMATION DATASET COMPLETED")
print("=" * 60)
print(f"Number of rows: {X_full.shape[0]}")
print(f"Number of features: {X_full.shape[1]}")
print(f"G3 present in X_full: {'G3' in X_full.columns}")
print(f"Parquet files saved to: {PROCESSED_DIR}")
print("=" * 60)

SESSION 20: FULL-INFORMATION DATASET COMPLETED
Number of rows: 395
Number of features: 41
G3 present in X_full: False
Parquet files saved to: /content/drive/MyDrive/GSSRP/data/processed


In [63]:
# Create the early-warning feature matrix by dropping G1 and G2
# We identify them dynamically to ensure accuracy
leak_cols = [c for c in X_full.columns if c in ("G1", "G2")]
X_early = X_full.drop(columns=leak_cols)

print("Full-information features shape:", X_full.shape)
print("Early-warning features shape:", X_early.shape)

Full-information features shape: (395, 41)
Early-warning features shape: (395, 39)


In [64]:
# Create the early-warning feature matrix
# We explicitly remove G1 and G2 to prevent "look-ahead" bias
leak_cols = [c for c in X_full.columns if c in ("G1", "G2")]
X_early = X_full.drop(columns=leak_cols)

print("Original full-information features:", X_full.shape[1])
print("Early-warning features:", X_early.shape[1])

# Verification
assert "G1" not in X_early.columns, "❌ G1 still present in X_early!"
assert "G2" not in X_early.columns, "❌ G2 still present in X_early!"
print("✅ Verification passed: Early-warning dataset is ready.")

Original full-information features: 41
Early-warning features: 39
✅ Verification passed: Early-warning dataset is ready.


In [65]:
# Save X_early and y as Parquet and CSV
X_early.to_parquet(PROCESSED_DIR / "X_early.parquet")
y.to_frame().to_parquet(PROCESSED_DIR / "y_early.parquet")

X_early.to_csv(PROCESSED_DIR / "X_early.csv", index=False)
y.to_csv(PROCESSED_DIR / "y_early.csv", index=False)

print("✅ Session 21 artifacts saved to data/processed/")

✅ Session 21 artifacts saved to data/processed/


In [66]:
import json

# 1. Create the Manifest (JSON)
manifest = {
    "dataset": "Early-Warning",
    "features_excluded": ["G1", "G2", "G3"],
    "total_features": int(X_early.shape[1]),
    "total_observations": int(X_early.shape[0])
}

with open(PROCESSED_DIR / "session21_early_warning_manifest.json", "w") as f:
    json.dump(manifest, f, indent=4)

# 2. Create the Artifact Summary (CSV)
summary_data = {
    "File": ["X_early.parquet", "y_early.parquet", "X_early.csv", "y_early.csv"],
    "Rows": [X_early.shape[0]] * 4,
    "Columns": [X_early.shape[1], 1, X_early.shape[1], 1]
}
pd.DataFrame(summary_data).to_csv(PROCESSED_DIR / "session21_early_warning_artifact_summary.csv", index=False)

print("✅ Manifest and Summary artifacts created.")

✅ Manifest and Summary artifacts created.


In [67]:
from pathlib import Path
import pandas as pd

# Define path (ensure this matches your Drive structure)
DRIVE_BASE = Path("/content/drive/MyDrive/GSSRP")
PROCESSED_DIR = DRIVE_BASE / "data" / "processed"

# Create the directory if it's missing
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Save the files
X_early.to_parquet(PROCESSED_DIR / "X_early.parquet")
y.to_frame().to_parquet(PROCESSED_DIR / "y_early.parquet")
X_early.to_csv(PROCESSED_DIR / "X_early.csv", index=False)
y.to_csv(PROCESSED_DIR / "y_early.csv", index=False)

print(f"✅ Files saved to: {PROCESSED_DIR}")

✅ Files saved to: /content/drive/MyDrive/GSSRP/data/processed


In [68]:
# Run this in Colab to ensure they are saved to your Drive
X_early.to_parquet(PROCESSED_DIR / "X_early.parquet")
y.to_frame().to_parquet(PROCESSED_DIR / "y_early.parquet")
X_early.to_csv(PROCESSED_DIR / "X_early.csv", index=False)
y.to_csv(PROCESSED_DIR / "y_early.csv", index=False)

In [69]:
import json
import pandas as pd
from pathlib import Path

# Ensure paths match your Google Drive setup
DRIVE_BASE = Path("/content/drive/MyDrive/GSSRP")
PROCESSED_DIR = DRIVE_BASE / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# 1. Generate the Manifest (JSON)
# Records metadata for the early-warning build
manifest = {
    "dataset": "Early-Warning",
    "features_excluded": ["G1", "G2", "G3"],
    "total_features": int(X_early.shape[1]),
    "total_observations": int(X_early.shape[0]),
    "status": "Validated"
}

with open(PROCESSED_DIR / "session21_early_warning_manifest.json", "w") as f:
    json.dump(manifest, f, indent=4)

# 2. Generate the Artifact Summary (CSV)
summary_data = {
    "File": ["X_early.parquet", "y_early.parquet", "X_early.csv", "y_early.csv"],
    "Rows": [X_early.shape[0]] * 4,
    "Columns": [X_early.shape[1], 1, X_early.shape[1], 1]
}
pd.DataFrame(summary_data).to_csv(PROCESSED_DIR / "session21_early_warning_artifact_summary.csv", index=False)

print("✅ session21_early_warning_manifest.json and artifact_summary.csv created.")

✅ session21_early_warning_manifest.json and artifact_summary.csv created.


# GSSRP 2026 — Session 22
## Section 3: Train/Test Split
This section creates reproducible training and test datasets for the
full-information and early-warning modeling scenarios.
Both scenarios use the same 80/20 split and the same random seed so that
their model results can be compared fairly.

In [70]:
from sklearn.model_selection import train_test_split

# Create reproducible splits for both feature sets
# Scenarios: Full-Information (_f) and Early-Warning (_e)
Xtr_f, Xte_f, ytr, yte = train_test_split(X_full, y, test_size=0.2, random_state=42)
Xtr_e, Xte_e, ytr_e, yte_e = train_test_split(X_early, y, test_size=0.2, random_state=42)

print("Full-Info Shapes:", Xtr_f.shape, Xte_f.shape)
print("Early-Warning Shapes:", Xtr_e.shape, Xte_e.shape)

# Verification
assert Xtr_f.shape[0] == ytr.shape[0], "Row mismatch in Full-Info training set"
assert Xtr_e.shape[0] == ytr_e.shape[0], "Row mismatch in Early-Warning training set"
print("✅ Train/test splits validated.")

Full-Info Shapes: (316, 41) (79, 41)
Early-Warning Shapes: (316, 39) (79, 39)
✅ Train/test splits validated.


In [71]:
def split_modeling_scenarios(X_full, X_early, y):
    from sklearn.model_selection import train_test_split

    # Full-Information Scenario
    Xtr_f, Xte_f, ytr, yte = train_test_split(X_full, y, test_size=0.2, random_state=42)

    # Early-Warning Scenario
    Xtr_e, Xte_e, ytr_e, yte_e = train_test_split(X_early, y, test_size=0.2, random_state=42)

    return (Xtr_f, Xte_f, ytr, yte), (Xtr_e, Xte_e, ytr_e, yte_e)

In [72]:
from sklearn.model_selection import train_test_split

def split_modeling_scenarios(X_full, X_early, y):
    """
    Performs reproducible train/test splits for both research scenarios.
    Returns: (Xtr_f, Xte_f, ytr, yte), (Xtr_e, Xte_e, ytr_e, yte_e)
    """
    # Full-Information Split
    Xtr_f, Xte_f, ytr, yte = train_test_split(X_full, y, test_size=0.2, random_state=42)

    # Early-Warning Split
    Xtr_e, Xte_e, ytr_e, yte_e = train_test_split(X_early, y, test_size=0.2, random_state=42)

    return (Xtr_f, Xte_f, ytr, yte), (Xtr_e, Xte_e, ytr_e, yte_e)

In [73]:
# src/preprocess.py
from sklearn.model_selection import train_test_split

def split_modeling_scenarios(X_full, X_early, y):
    """
    Performs consistent 80/20 splits for both Full-Info and Early-Warning data.
    """
    # Full-Information Scenario
    Xtr_f, Xte_f, ytr, yte = train_test_split(X_full, y, test_size=0.2, random_state=42)

    # Early-Warning Scenario
    Xtr_e, Xte_e, ytr_e, yte_e = train_test_split(X_early, y, test_size=0.2, random_state=42)

    return (Xtr_f, Xte_f, ytr, yte), (Xtr_e, Xte_e, ytr_e, yte_e)

In [77]:
import os

# List contents of the base drive to find your project
print("--- Root Drive Contents ---")
print(os.listdir('/content/drive/MyDrive/'))

# Once you identify the folder (e.g., GSSRP or GSSRP_2026),
# look inside it by updating this line:
# print(os.listdir('/content/drive/MyDrive/YOUR_FOLDER_NAME_HERE'))

--- Root Drive Contents ---
['Classroom', 'Miranda  birthday.gdoc', 'tye-czed-vsr - Jan 5, 2022 (3).pdf', 'tye-czed-vsr - Jan 5, 2022 (2).pdf', 'tye-czed-vsr - Jan 5, 2022 (1).pdf', 'tye-czed-vsr - Jan 5, 2022.pdf', 'tye-czed-vsr - Jan 6, 2022 (4).pdf', 'tye-czed-vsr - Jan 6, 2022 (3).pdf', 'tye-czed-vsr - Jan 6, 2022 (2).pdf', 'tye-czed-vsr - Jan 6, 2022 (1).pdf', 'tye-czed-vsr - Jan 6, 2022.pdf', 'tye-czed-vsr - Jan 11, 2022 (20).pdf', 'tye-czed-vsr - Jan 11, 2022 (19).pdf', 'tye-czed-vsr - Jan 11, 2022 (18).pdf', 'tye-czed-vsr - Jan 11, 2022 (17).pdf', 'tye-czed-vsr - Jan 11, 2022 (16).pdf', 'tye-czed-vsr - Jan 11, 2022 (15).pdf', 'tye-czed-vsr - Jan 11, 2022 (14).pdf', 'tye-czed-vsr - Jan 11, 2022 (13).pdf', 'tye-czed-vsr - Jan 11, 2022 (12).pdf', 'tye-czed-vsr - Jan 11, 2022 (11).pdf', 'tye-czed-vsr - Jan 11, 2022 (10).pdf', 'tye-czed-vsr - Jan 11, 2022 (9).pdf', 'tye-czed-vsr - Jan 11, 2022 (8).pdf', 'tye-czed-vsr - Jan 11, 2022 (7).pdf', 'tye-czed-vsr - Jan 11, 2022 (6).pdf', 't

In [78]:
# Check inside your GSSRP folder
print(os.listdir('/content/drive/MyDrive/GSSRP'))

['data']


In [81]:
try:
    from src.preprocess import split_modeling_scenarios
    print("Import successful! You can now proceed with Session 22 tasks.")
except ImportError as e:
    print(f"Import failed: {e}")

Import failed: No module named 'src'


In [83]:
from pathlib import Path
import math
import pandas as pd
from sklearn.model_selection import train_test_split
print("Required libraries imported successfully.")

Required libraries imported successfully.


In [84]:
from pathlib import Path

# --- Configuration: Define Project Paths ---
# Use consistent pathing for GSSRP project components
DRIVE_BASE = Path("/content/drive/MyDrive/GSSRP")
PROCESSED_DIR = DRIVE_BASE / "data" / "processed"

# --- Verification: Ensure environment is set up correctly ---
def verify_paths():
    print(f"--- Path Verification ---")
    print(f"Project Folder:    {DRIVE_BASE}")
    print(f"Processed Folder:  {PROCESSED_DIR}")
    print(f"Folder Exists:     {PROCESSED_DIR.exists()}")

    if not PROCESSED_DIR.exists():
        print("⚠️ Warning: The processed directory was not found. Please check your Drive path.")

verify_paths()

--- Path Verification ---
Project Folder:    /content/drive/MyDrive/GSSRP
Processed Folder:  /content/drive/MyDrive/GSSRP/data/processed
Folder Exists:     True


In [85]:
import pandas as pd
from pathlib import Path

# Ensure PROCESSED_DIR is defined globally or passed to the function
# Example: PROCESSED_DIR = Path("/content/drive/MyDrive/GSSRP/data/processed")

def load_processed_artifact(file_stem: str):
    """
    Load a processed dataset from Parquet or CSV.

    Parameters
    ----------
    file_stem : str
        Filename without the .parquet or .csv extension.

    Returns
    -------
    data : pandas.DataFrame
        Loaded dataset.
    format_used : str
        File format used.
    file_path : pathlib.Path
        Full path of the loaded file.
    """
    parquet_path = PROCESSED_DIR / f"{file_stem}.parquet"
    csv_path = PROCESSED_DIR / f"{file_stem}.csv"

    if parquet_path.exists():
        data = pd.read_parquet(parquet_path)
        return data, "Parquet", parquet_path

    if csv_path.exists():
        data = pd.read_csv(csv_path)
        return data, "CSV", csv_path

    raise FileNotFoundError(
        f"Neither {parquet_path.name} nor {csv_path.name} was found in:\n{PROCESSED_DIR}"
    )

print("✅ Dataset-loading function created.")

✅ Dataset-loading function created.


In [86]:
def validate_modeling_data(X_full, X_early, y):
    """
    Performs critical integrity checks on feature sets and target before splitting.
    """
    assert isinstance(X_full, pd.DataFrame), "X_full must be a pandas DataFrame."
    assert isinstance(X_early, pd.DataFrame), "X_early must be a pandas DataFrame."
    assert isinstance(y, pd.Series), "y must be a pandas Series."

    # Check for target leakage
    assert "G3" not in X_full.columns, "Target leakage detected: G3 is present in X_full."
    assert "G3" not in X_early.columns, "Target leakage detected: G3 is present in X_early."
    assert y.name == "G3", f"Target should be named 'G3', but it is named '{y.name}'."

    # Check structural consistency
    assert len(X_full) == len(X_early), "X_full and X_early have different row counts."
    assert len(X_full) == len(y), "X_full and y have different row counts."
    assert X_full.index.equals(X_early.index), "X_full and X_early do not have identical indexes."
    assert X_full.index.equals(y.index), "X_full and y do not have identical indexes."
    assert X_full.index.is_unique, "X_full contains duplicate row indexes."

    # Check feature subset logic
    assert set(X_early.columns).issubset(set(X_full.columns)), "X_early contains columns not in X_full."
    assert "G1" in X_full.columns, "G1 is missing from the full-information dataset."
    assert "G2" in X_full.columns, "G2 is missing from the full-information dataset."
    assert "G1" not in X_early.columns, "G1 is still present in the early-warning dataset."
    assert "G2" not in X_early.columns, "G2 is still present in the early-warning dataset."

    print("✅ Input validation passed successfully.")

# Usage:
# validate_modeling_data(X_full, X_early, y)

In [87]:
print("DATASETS BEFORE SPLITTING")
print("-" * 45)

print(f"X_full shape:              {X_full.shape}")
print(f"X_early shape:             {X_early.shape}")
print(f"y shape:                   {y.shape}")

print("-" * 45)
print(f"Full-information features: {X_full.shape[1]}")
print(f"Early-warning features:    {X_early.shape[1]}")
print(f"Feature-count difference:  {X_full.shape[1] - X_early.shape[1]}")

DATASETS BEFORE SPLITTING
---------------------------------------------
X_full shape:              (395, 41)
X_early shape:             (395, 39)
y shape:                   (395,)
---------------------------------------------
Full-information features: 41
Early-warning features:    39
Feature-count difference:  2


In [88]:
# --- Configuration: Split Parameters ---
TEST_SIZE = 0.20
RANDOM_STATE = 42

print("--- Split Configuration ---")
print(f"Training fraction: {1 - TEST_SIZE:.0%}")
print(f"Test fraction:     {TEST_SIZE:.0%}")
print(f"Random state:      {RANDOM_STATE}")

--- Split Configuration ---
Training fraction: 80%
Test fraction:     20%
Random state:      42


In [89]:
# --- Execution: Split Full-Information Scenario ---
Xtr_f, Xte_f, ytr, yte = train_test_split(
    X_full,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True
)

print("✅ Full-information split completed.")
print("-" * 35)
print(f"Training features: {Xtr_f.shape}")
print(f"Test features:     {Xte_f.shape}")
print(f"Training target:   {ytr.shape}")
print(f"Test target:       {yte.shape}")

✅ Full-information split completed.
-----------------------------------
Training features: (316, 41)
Test features:     (79, 41)
Training target:   (316,)
Test target:       (79,)


In [90]:
# --- Execution: Split Early-Warning Scenario ---
Xtr_e, Xte_e, ytr_e, yte_e = train_test_split(
    X_early,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True
)

print("✅ Early-warning split completed.")
print("-" * 35)
print(f"Training features: {Xtr_e.shape}")
print(f"Test features:     {Xte_e.shape}")
print(f"Training target:   {ytr_e.shape}")
print(f"Test target:       {yte_e.shape}")

✅ Early-warning split completed.
-----------------------------------
Training features: (316, 39)
Test features:     (79, 39)
Training target:   (316,)
Test target:       (79,)


In [91]:
# --- Verification: Output Split Shapes ---
print(f"Full-Information Split Shapes:")
print(f"  Training: {Xtr_f.shape}")
print(f"  Testing:  {Xte_f.shape}")

print(f"\nEarly-Warning Split Shapes:")
print(f"  Training: {Xtr_e.shape}")
print(f"  Testing:  {Xte_e.shape}")

Full-Information Split Shapes:
  Training: (316, 41)
  Testing:  (79, 41)

Early-Warning Split Shapes:
  Training: (316, 39)
  Testing:  (79, 39)


In [92]:
# --- Verification: Index Consistency ---
same_train_indices = Xtr_f.index.equals(Xtr_e.index)
same_test_indices = Xte_f.index.equals(Xte_e.index)
same_training_targets = ytr.index.equals(ytr_e.index)
same_test_targets = yte.index.equals(yte_e.index)

print("--- Index Consistency Check ---")
print(f"Same training row indices:     {same_train_indices}")
print(f"Same test row indices:         {same_test_indices}")
print(f"Same training target indices:  {same_training_targets}")
print(f"Same test target indices:      {same_test_targets}")

--- Index Consistency Check ---
Same training row indices:     True
Same test row indices:         True
Same training target indices:  True
Same test target indices:      True


In [93]:
# --- Validation: Index Consistency ---
assert same_train_indices, "The two scenarios use different training rows."
assert same_test_indices, "The two scenarios use different test rows."
assert same_training_targets, "The training target indexes do not match."
assert same_test_targets, "The test target indexes do not match."

pd.testing.assert_series_equal(ytr, ytr_e)
pd.testing.assert_series_equal(yte, yte_e)

print("✅ Both modeling scenarios use identical training and test observations.")

✅ Both modeling scenarios use identical training and test observations.


In [94]:
# --- Validation: No Data Leakage ---
training_index_set = set(Xtr_f.index)
test_index_set = set(Xte_f.index)
original_index_set = set(X_full.index)

assert training_index_set.isdisjoint(test_index_set), \
    "Data leakage detected: some observations appear in both sets."
assert (training_index_set | test_index_set) == original_index_set, \
    "The split does not contain every original observation."

print("✅ Training and test sets do not overlap.")
print("✅ Every original observation is included exactly once.")

✅ Training and test sets do not overlap.
✅ Every original observation is included exactly once.


In [95]:
import math
# --- Validation: Row Count Proportions ---
total_rows = len(X_full)
expected_test_rows = math.ceil(total_rows * TEST_SIZE)
expected_training_rows = total_rows - expected_test_rows

assert len(Xtr_f) == expected_training_rows
assert len(Xtr_e) == expected_training_rows
assert len(ytr) == expected_training_rows
assert len(Xte_f) == expected_test_rows
assert len(Xte_e) == expected_test_rows
assert len(yte) == expected_test_rows

print(f"Total observations:      {total_rows}")
print(f"Expected training rows:  {expected_training_rows}")
print(f"Actual training rows:    {len(Xtr_f)}")
print(f"Expected test rows:      {expected_test_rows}")
print(f"Actual test rows:        {len(Xte_f)}")
print("✅ 80/20 row-count verification passed.")

Total observations:      395
Expected training rows:  316
Actual training rows:    316
Expected test rows:      79
Actual test rows:        79
✅ 80/20 row-count verification passed.


In [96]:
# --- Validation: Reproducibility ---
Xtr_f_repeat, Xte_f_repeat, ytr_repeat, yte_repeat = train_test_split(
    X_full, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, shuffle=True
)

pd.testing.assert_frame_equal(Xtr_f, Xtr_f_repeat)
pd.testing.assert_frame_equal(Xte_f, Xte_f_repeat)
pd.testing.assert_series_equal(ytr, ytr_repeat)
pd.testing.assert_series_equal(yte, yte_repeat)

print("✅ Reproducibility check passed.")
print("✅ Running the split again with random_state=42 produces exactly the same result.")

✅ Reproducibility check passed.
✅ Running the split again with random_state=42 produces exactly the same result.


In [97]:
split_summary = pd.DataFrame({
    "Scenario": ["Full-information", "Early-warning"],
    "Training_Rows": [Xtr_f.shape[0], Xtr_e.shape[0]],
    "Test_Rows": [Xte_f.shape[0], Xte_e.shape[0]],
    "Feature_Count": [Xtr_f.shape[1], Xtr_e.shape[1]],
    "Target": [ytr.name, ytr.name],
    "Test_Fraction": [TEST_SIZE, TEST_SIZE],
    "Random_State": [RANDOM_STATE, RANDOM_STATE]
})
display(split_summary)

,Scenario,Training_Rows,Test_Rows,Feature_Count,Target,Test_Fraction,Random_State
0,Full-information,316,79,41,G3,0.2,42
1,Early-warning,316,79,39,G3,0.2,42


In [98]:
validation_table = pd.DataFrame({
    "Validation_Check": [
        "Full and early row counts match", "Training indexes are identical",
        "Test indexes are identical", "Training targets are identical",
        "Test targets are identical", "Training and test sets do not overlap",
        "All observations are retained", "Repeated split is reproducible"
    ],
    "Result": [
        len(X_full) == len(X_early), Xtr_f.index.equals(Xtr_e.index),
        Xte_f.index.equals(Xte_e.index), ytr.equals(ytr_e),
        yte.equals(yte_e), training_index_set.isdisjoint(test_index_set),
        (training_index_set | test_index_set) == original_index_set,
        Xte_f.equals(Xte_f_repeat)
    ]
})
display(validation_table)

,Validation_Check,Result
0,Full and early row counts match,True
1,Training indexes are identical,True
2,Test indexes are identical,True
3,Training targets are identical,True
4,Test targets are identical,True
5,Training and test sets do not overlap,True
6,All observations are retained,True
7,Repeated split is reproducible,True


In [100]:
# --- Updated Save Utility ---
def save_artifact(df_or_series, name):
    # If it's a Series, convert to DataFrame first to enable to_parquet
    if isinstance(df_or_series, pd.Series):
        df_to_save = df_or_series.to_frame()
    else:
        df_to_save = df_or_series

    path = PROCESSED_DIR / f"{name}.parquet"
    df_to_save.to_parquet(path)
    print(f"✅ Saved: {name} to {path}")

# --- Retry Saving All Objects ---
objects_to_save = {
    "Xtr_f": Xtr_f, "Xte_f": Xte_f,
    "Xtr_e": Xtr_e, "Xte_e": Xte_e,
    "ytr": ytr, "yte": yte
}

for name, obj in objects_to_save.items():
    save_artifact(obj, name)

✅ Saved: Xtr_f to /content/drive/MyDrive/GSSRP/data/processed/Xtr_f.parquet
✅ Saved: Xte_f to /content/drive/MyDrive/GSSRP/data/processed/Xte_f.parquet
✅ Saved: Xtr_e to /content/drive/MyDrive/GSSRP/data/processed/Xtr_e.parquet
✅ Saved: Xte_e to /content/drive/MyDrive/GSSRP/data/processed/Xte_e.parquet
✅ Saved: ytr to /content/drive/MyDrive/GSSRP/data/processed/ytr.parquet
✅ Saved: yte to /content/drive/MyDrive/GSSRP/data/processed/yte.parquet


### Summary

In machine learning research, using the same `random_state` and indexing across multiple modeling scenarios is a standard best practice. It ensures that the subset of data used for training and testing remains constant regardless of how many times the code is executed or which feature set is being analyzed.

### Interpretation

* **Reproducibility:** Machine learning models rely on pseudo-random number generators to shuffle and partition data. By fixing the `random_state` (e.g., to `42`), you force the computer to produce the exact same sequence of "random" numbers. This ensures that anyone who runs your code—including yourself in future sessions—will arrive at the exact same training and testing partitions, making your research findings verifiable and transparent.
* **Comparability:** This is arguably the most critical aspect of your project. By using identical row indices across your Full-Information (`X_full`) and Early-Warning (`X_early`) datasets, you ensure that both models are tested on the exact same student populations. If the splits differed, any discrepancy in model performance could be incorrectly attributed to the features themselves, when it might actually be due to one model being tested on "easier" or "harder" student data than the other. Synchronization allows for a true "apples-to-apples" comparison of predictive power.

### Recommendation

* **Test Fraction:** I recommend using a test fraction of **0.20 (20%)**.
* **Rationale:** This ratio strikes a professional balance between training and evaluation. It provides enough data in the training set (80%) for the model to effectively learn complex patterns in student performance, while leaving a sufficiently large testing subset (20%) to ensure that your performance metrics (like accuracy, precision, or RMSE) are statistically significant and representative of the real world. Anything significantly smaller may lead to unreliable metrics, while a larger test set often starves the model of the data it needs to perform at its peak.

# 4. Prompt-Engineered Explanation Task
This task explains why the full-information and early-warning scenarios
must use the same train/test split and the same random seed.

Role: Act as an evaluation-design assistant.
Task: Explain why the same random_state and the same train/test split are
used across the full-information and early-warning modeling scenarios.
Context:
The project compares two feature sets:
1. Full-information scenario:
Includes all approved predictors, including G1 and G2.
2. Early-warning scenario:
Excludes G1 and G2 so that predictions can be made earlier.
Both scenarios predict the same target, G3.
Requirements:
1. Explain reproducibility.
2. Explain comparability.
3. Explain why the same students must appear in the training and test sets
for both scenarios.
4. Explain what could go wrong if different splits were used.
5. Recommend an appropriate test fraction for this dataset.
6. Use clear language suitable for a student machine-learning research
project.


Output format:

Summary

Interpretation


Recommendation

To ensure a fair and scientifically valid comparison between your modeling scenarios, it is essential that both models be evaluated on the same subset of student data. This is achieved by synchronizing the data partitions across both scenarios.

### Comparison Framework

The comparison is structured to evaluate the predictive impact of removing mid-semester progress indicators. The modeling scenarios are defined as follows:

| Scenario | Feature Set | Purpose |
| --- | --- | --- |
| **Full-information** | $X_{full}$ | Predict $G3$ using all available information, including $G1$ and $G2$. |
| **Early-warning** | $X_{early}$ | Predict $G3$ without using $G1$ and $G2$. |

The target variable $y$ is shared across both scenarios and represents the final grade, $G3$.

### Methodological Requirements

To measure the specific effect of removing the features $G1$ and $G2$, the experimental design must adhere to the following constraints:

* **Identical Population Partitioning:** The set of students assigned to the training set and the test set must be identical across both scenarios. If $i$ represents an index in the set of all students $S$, then:

$$S_{train, full} = S_{train, early}$$


$$S_{test, full} = S_{test, early}$$


* **Synchronized Randomization:** By utilizing a consistent $random\_state$ during the $train\_test\_split$ process, we ensure that the indices for the training and testing partitions are deterministic and repeatable:

$$Index(X_{tr, f}) = Index(X_{tr, e})$$


$$Index(X_{te, f}) = Index(X_{te, e})$$



By maintaining these constraints, any observed difference in model performance—quantified by metrics such as Mean Squared Error (MSE) or $R^2$—can be attributed solely to the exclusion of the features $G1$ and $G2$, rather than variations in the data samples used for training or validation.

##Summary
The same random_state and the same train/test split are used for the full-information and early-warning
scenarios to make the experiment reproducible and the model comparison fair. The random_state
controls the random assignment of observations to the training and test sets. When the same value, such
as random_state=42, is used, the split can be recreated exactly each time the notebook is run. Using the
same split also ensures that both scenarios are trained and evaluated using the same students.
##Interpretation
##Reproducibility
 means that another student or researcher can run the same code and obtain the same
training and test observations. Without a fixed random_state, the selected training and test students

##GSSRP 2026 - Kean University | Predicting Student Performance Using Machine Learning Session 22/48 - Week 2
could change each time the code is executed. This could cause model performance metrics to change
even when the model and features have not changed.
##Comparability
means that the full-information and early-warning models are evaluated under equivalent
conditions. The main experimental difference should be the feature set. The full-information scenario
contains G1 and G2, while the early-warning scenario excludes them. All other evaluation conditions
should remain constant.
Using identical row indices ensures that both models make predictions for the same students in the test
set. Some students may be easier to predict than others. If the full-information model were evaluated on
one group of students and the early-warning model were evaluated on another group, differences in
model performance might be caused by differences between the student groups rather than by the
presence or absence of G1 and G2.
Different splits would therefore introduce an additional source of variation into the experiment. For
example, one test set might contain more students with unusual academic patterns, missing information,
high absences, or extreme final grades. A model evaluated on that group might appear less accurate even
though its feature set is not the real cause of the lower performance. This would make the comparison
less reliable and potentially misleading.
##Recommendation
Use the same random_state, the same test fraction, and identical training and test row indices for both
modeling scenarios. A test fraction of 0.20, or 20%, is appropriate for this project. It provides
approximately 80% of the observations for model training while reserving 20% for independent
evaluation.

The split should be created using:
train_test_split(
X,
y,
test_size=0.20,
random_state=42
)
After splitting, confirm that the full-information and early-warning scenarios have identical training
indexes and identical test indexes. This design ensures that observed performance differences are
primarily attributable to the different feature sets rather than to different student samples.
##Summary
The same random_state and the same train/test split are used for the full-information and early-warning
scenarios to make the experiment reproducible and the model comparison fair. The random_state
controls the random assignment of observations to the training and test sets. When the same value, such
as random_state=42, is used, the split can be recreated exactly each time the notebook is run. Using the
same split also ensures that both scenarios are trained and evaluated using the same students.
##Interpretation
##Reproducibility means that another student or researcher can run the same code and obtain the same
training and test observations. Without a fixed random_state, the selected training and test students

##GSSRP 2026 - Kean University | Predicting Student Performance Using Machine Learning Session 22/48 - Week 2
could change each time the code is executed. This could cause model performance metrics to change
even when the model and features have not changed.
##Comparability means that the full-information and early-warning models are evaluated under equivalent
conditions. The main experimental difference should be the feature set. The full-information scenario
contains G1 and G2, while the early-warning scenario excludes them. All other evaluation conditions
should remain constant.
Using identical row indices ensures that both models make predictions for the same students in the test
set. Some students may be easier to predict than others. If the full-information model were evaluated on
one group of students and the early-warning model were evaluated on another group, differences in
model performance might be caused by differences between the student groups rather than by the
presence or absence of G1 and G2.
Different splits would therefore introduce an additional source of variation into the experiment. For
example, one test set might contain more students with unusual academic patterns, missing information,
high absences, or extreme final grades. A model evaluated on that group might appear less accurate even
though its feature set is not the real cause of the lower performance. This would make the comparison
less reliable and potentially misleading.
##Recommendation
Use the same random_state, the same test fraction, and identical training and test row indices for both
modeling scenarios. A test fraction of 0.20, or 20%, is appropriate for this project. It provides
approximately 80% of the observations for model training while reserving 20% for independent
evaluation.
The split should be created using:
train_test_split(
X,
y,
test_size=0.20,
random_state=42
)
After splitting, confirm that the full-information and early-warning scenarios have identical training
indexes and identical test indexes. This design ensures that observed performance differences are
primarily attributable to the different feature sets rather than to different student samples.

### Research Interpretation
The comparison between `X_full` and `X_early` is a controlled experiment.
- The target remains the same: `G3`.
- The students remain the same.

GSSRP 2026 - Kean University | Predicting Student Performance Using Machine Learning Session 22/48 - Week 2
- The training and test proportions remain the same.
- The random seed remains the same.
- Only the available predictors change.
Therefore, differences in model performance can be interpreted as evidence
about the value of G1 and G2 rather than evidence about differences between
the samples.

In [101]:
# --- Verification: Dimensions and Row Order ---
required_objects = ["Xtr_f", "Xte_f", "Xtr_e", "Xte_e", "ytr", "yte"]
missing_objects = [obj for obj in required_objects if obj not in globals()]

if missing_objects:
    raise NameError(f"Missing objects: {', '.join(missing_objects)}. Run Section 3 first.")

print("✅ All required train/test objects are available.")

print(f"Full-info test shape:  {Xte_f.shape}")
print(f"Early-warn test shape: {Xte_e.shape}")

✅ All required train/test objects are available.
Full-info test shape:  (79, 41)
Early-warn test shape: (79, 39)


In [102]:
# --- Verification: Index Consistency ---
identical_test_indices = Xte_f.index.equals(Xte_e.index)
print(f"Do test row indices match? {identical_test_indices}")

# Create side-by-side comparison
test_index_comparison = pd.DataFrame({
    "Test_Position": range(1, len(Xte_f) + 1),
    "Full_Information_Index": Xte_f.index.tolist(),
    "Early_Warning_Index": Xte_e.index.tolist()
})
test_index_comparison["Indices_Match"] = (
    test_index_comparison["Full_Information_Index"] == test_index_comparison["Early_Warning_Index"]
)

display(test_index_comparison)
print(f"✅ All test-set row indices match: {test_index_comparison['Indices_Match'].all()}")

Do test row indices match? True


,Test_Position,Full_Information_Index,Early_Warning_Index,Indices_Match
0,1,78,78,True
1,2,371,371,True
2,3,248,248,True
3,4,55,55,True
4,5,390,390,True
...,...,...,...,...
74,75,364,364,True
75,76,82,82,True
76,77,114,114,True
77,78,3,3,True


✅ All test-set row indices match: True


In [103]:
# --- Verification: Set Theory Checks ---
full_test_index_set = set(Xte_f.index)
early_test_index_set = set(Xte_e.index)

only_in_full = full_test_index_set - early_test_index_set
only_in_early = early_test_index_set - full_test_index_set

print(f"Indices only in full: {sorted(only_in_full)}")
print(f"Indices only in early: {sorted(only_in_early)}")

assert Xte_f.index.equals(yte.index), "Full test indices mismatch target"
assert Xte_e.index.equals(yte.index), "Early test indices mismatch target"

Indices only in full: []
Indices only in early: []


In [104]:
# --- Verification: Data Correspondence ---
pd.testing.assert_frame_equal(Xte_f, X_full.loc[Xte_f.index])
pd.testing.assert_frame_equal(Xte_e, X_early.loc[Xte_e.index])
pd.testing.assert_series_equal(yte, y.loc[yte.index])

print(f"Feature count diff: {Xte_f.shape[1] - Xte_e.shape[1]}")
print(f"G1/G2 excluded in early-warning: {'G1' not in Xte_e.columns and 'G2' not in Xte_e.columns}")

Feature count diff: 2
G1/G2 excluded in early-warning: True


In [105]:
# --- Final Validation Table ---
student_activity_summary = pd.DataFrame({
    "Check": [
        "Same number of test rows", "Identical test index order",
        "Identical test index set", "No indices only in full",
        "No indices only in early", "Full test indices match target",
        "Early test indices match target", "G1 excluded", "G2 excluded"
    ],
    "Result": [
        len(Xte_f) == len(Xte_e), Xte_f.index.equals(Xte_e.index),
        full_test_index_set == early_test_index_set, len(only_in_full) == 0,
        len(only_in_early) == 0, Xte_f.index.equals(yte.index),
        Xte_e.index.equals(yte.index), "G1" not in Xte_e.columns, "G2" not in Xte_e.columns
    ]
})

display(student_activity_summary)
assert student_activity_summary["Result"].all()
print("=" * 68)
print("SESSION 22 SECTION 5 COMPLETED SUCCESSFULLY")
print("=" * 68)

,Check,Result
0,Same number of test rows,True
1,Identical test index order,True
2,Identical test index set,True
3,No indices only in full,True
4,No indices only in early,True
5,Full test indices match target,True
6,Early test indices match target,True
7,G1 excluded,True
8,G2 excluded,True


SESSION 22 SECTION 5 COMPLETED SUCCESSFULLY


## Student Conclusion
The full-information and early-warning scenarios use identical row indices
in the test set. The row-by-row comparison produced only `True` values, and
there were zero nonmatching observations.
This confirms that both models will be evaluated using the same students.

GSSRP 2026 - Kean University | Predicting Student Performance Using Machine Learning Session 22/48 - Week 2
The only intentional difference between the scenarios is that the
full-information feature set includes G1 and G2, while the early-warning
feature set excludes them. Therefore, future differences in model
performance can be attributed more reliably to the feature sets rather than
to differences in the test samples.

# 6. Output Artifact
## Modeling-Ready Train/Test Datasets
This section validates and saves the training and test datasets for the
full-information and early-warning modeling scenarios.
The feature matrices are saved separately, while both scenarios share the

In [106]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

# Setup path to GSSRP project
DRIVE_BASE = Path("/content/drive/MyDrive/GSSRP")
OUTPUT_DIR = DRIVE_BASE / "data" / "processed" / "session22_splits"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Artifact-saving libraries imported.")
print(f"Output folder: {OUTPUT_DIR}")
print(f"Output folder exists: {OUTPUT_DIR.exists()}")

✅ Artifact-saving libraries imported.
Output folder: /content/drive/MyDrive/GSSRP/data/processed/session22_splits
Output folder exists: True


In [107]:
# Check object types
assert isinstance(Xtr_f, pd.DataFrame), "Xtr_f must be a DataFrame."
assert isinstance(ytr, pd.Series), "ytr must be a Series."

# Row count consistency
assert len(Xtr_f) == len(Xtr_e) == len(ytr), "Training row counts mismatch."
assert len(Xte_f) == len(Xte_e) == len(yte), "Test row counts mismatch."

# Row index alignment
assert Xtr_f.index.equals(Xtr_e.index) and Xtr_f.index.equals(ytr.index), "Training indices mismatch."
assert Xte_f.index.equals(Xte_e.index) and Xte_f.index.equals(yte.index), "Test indices mismatch."

# Target Leakage & Feature Content
feature_objects = {"Xtr_f": Xtr_f, "Xte_f": Xte_f, "Xtr_e": Xtr_e, "Xte_e": Xte_e}
for name, df in feature_objects.items():
    assert "G3" not in df.columns, f"Target leakage detected in {name}."

# Confirm Scenario Definitions
assert "G1" in Xtr_f.columns and "G2" in Xtr_f.columns, "G1/G2 missing from Full."
assert "G1" not in Xtr_e.columns and "G2" not in Xtr_e.columns, "G1/G2 present in Early."

print("✅ All integrity validations passed.")

✅ All integrity validations passed.


In [108]:
# Check for nulls and non-numeric types
missing_value_summary = pd.DataFrame({
    "Dataset": ["Xtr_f", "Xte_f", "Xtr_e", "Xte_e", "ytr", "yte"],
    "Missing_Values": [Xtr_f.isna().sum().sum(), Xte_f.isna().sum().sum(),
                       Xtr_e.isna().sum().sum(), Xte_e.isna().sum().sum(),
                       ytr.isna().sum(), yte.isna().sum()]
})
display(missing_value_summary)
assert (missing_value_summary["Missing_Values"] == 0).all(), "Missing values found!"

# Numeric and Infinity check
for name, df in feature_objects.items():
    non_numeric = df.select_dtypes(exclude=["number", "bool"]).columns.tolist()
    assert len(non_numeric) == 0, f"{name} contains non-numeric columns: {non_numeric}"

    numeric_array = df.to_numpy(dtype=float)
    assert not np.isinf(numeric_array).any(), f"Infinite values found in {name}."

print("✅ Data is clean, numeric, and finite.")

,Dataset,Missing_Values
0,Xtr_f,0
1,Xte_f,0
2,Xtr_e,0
3,Xte_e,0
4,ytr,0
5,yte,0


✅ Data is clean, numeric, and finite.


In [109]:
# Standardize target names
ytr = ytr.copy(); ytr.name = "G3"
yte = yte.copy(); yte.name = "G3"

# Summary of Artifacts
artifact_shapes = pd.DataFrame({
    "Artifact": ["X_train_full", "X_test_full", "X_train_early", "X_test_early", "y_train", "y_test"],
    "Rows": [Xtr_f.shape[0], Xte_f.shape[0], Xtr_e.shape[0], Xte_e.shape[0], ytr.shape[0], yte.shape[0]],
    "Columns": [Xtr_f.shape[1], Xte_f.shape[1], Xtr_e.shape[1], Xte_e.shape[1], 1, 1]
})
display(artifact_shapes)

# Parquet Paths
parquet_paths = {
    "X_train_full": OUTPUT_DIR / "X_train_full.parquet",
    "X_test_full": OUTPUT_DIR / "X_test_full.parquet",
    "X_train_early": OUTPUT_DIR / "X_train_early.parquet",
    "X_test_early": OUTPUT_DIR / "X_test_early.parquet",
    "y_train": OUTPUT_DIR / "y_train.parquet",
    "y_test": OUTPUT_DIR / "y_test.parquet"
}

for name, path in parquet_paths.items():
    print(f"{name}: {path}")

,Artifact,Rows,Columns
0,X_train_full,316,41
1,X_test_full,79,41
2,X_train_early,316,39
3,X_test_early,79,39
4,y_train,316,1
5,y_test,79,1


X_train_full: /content/drive/MyDrive/GSSRP/data/processed/session22_splits/X_train_full.parquet
X_test_full: /content/drive/MyDrive/GSSRP/data/processed/session22_splits/X_test_full.parquet
X_train_early: /content/drive/MyDrive/GSSRP/data/processed/session22_splits/X_train_early.parquet
X_test_early: /content/drive/MyDrive/GSSRP/data/processed/session22_splits/X_test_early.parquet
y_train: /content/drive/MyDrive/GSSRP/data/processed/session22_splits/y_train.parquet
y_test: /content/drive/MyDrive/GSSRP/data/processed/session22_splits/y_test.parquet


In [110]:
# --- Save All Artifacts to Parquet ---
def save_artifact(obj, name, path):
    # Convert Series to DataFrame if necessary
    to_save = obj.to_frame() if isinstance(obj, pd.Series) else obj
    to_save.to_parquet(path)
    print(f"✅ Saved: {name} to {path}")

# Perform the save for all objects
for name, path in parquet_paths.items():
    if name == "X_train_full": save_artifact(Xtr_f, name, path)
    if name == "X_test_full": save_artifact(Xte_f, name, path)
    if name == "X_train_early": save_artifact(Xtr_e, name, path)
    if name == "X_test_early": save_artifact(Xte_e, name, path)
    if name == "y_train": save_artifact(ytr, name, path)
    if name == "y_test": save_artifact(yte, name, path)

✅ Saved: X_train_full to /content/drive/MyDrive/GSSRP/data/processed/session22_splits/X_train_full.parquet
✅ Saved: X_test_full to /content/drive/MyDrive/GSSRP/data/processed/session22_splits/X_test_full.parquet
✅ Saved: X_train_early to /content/drive/MyDrive/GSSRP/data/processed/session22_splits/X_train_early.parquet
✅ Saved: X_test_early to /content/drive/MyDrive/GSSRP/data/processed/session22_splits/X_test_early.parquet
✅ Saved: y_train to /content/drive/MyDrive/GSSRP/data/processed/session22_splits/y_train.parquet
✅ Saved: y_test to /content/drive/MyDrive/GSSRP/data/processed/session22_splits/y_test.parquet


In [112]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

# Setup path to GSSRP project
DRIVE_BASE = Path("/content/drive/MyDrive/GSSRP")
OUTPUT_DIR = DRIVE_BASE / "data" / "processed" / "session22_splits"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Artifact-saving libraries imported.")
print(f"Output folder: {OUTPUT_DIR}")
print(f"Output folder exists: {OUTPUT_DIR.exists()}")

✅ Artifact-saving libraries imported.
Output folder: /content/drive/MyDrive/GSSRP/data/processed/session22_splits
Output folder exists: True


In [113]:
# Check object types
assert isinstance(Xtr_f, pd.DataFrame), "Xtr_f must be a DataFrame."
assert isinstance(ytr, pd.Series), "ytr must be a Series."

# Row count consistency
assert len(Xtr_f) == len(Xtr_e) == len(ytr), "Training row counts mismatch."
assert len(Xte_f) == len(Xte_e) == len(yte), "Test row counts mismatch."

# Row index alignment
assert Xtr_f.index.equals(Xtr_e.index) and Xtr_f.index.equals(ytr.index), "Training indices mismatch."
assert Xte_f.index.equals(Xte_e.index) and Xte_f.index.equals(yte.index), "Test indices mismatch."

# Target Leakage & Feature Content
feature_objects = {"Xtr_f": Xtr_f, "Xte_f": Xte_f, "Xtr_e": Xtr_e, "Xte_e": Xte_e}
for name, df in feature_objects.items():
    assert "G3" not in df.columns, f"Target leakage detected in {name}."

# Confirm Scenario Definitions
assert "G1" in Xtr_f.columns and "G2" in Xtr_f.columns, "G1/G2 missing from Full."
assert "G1" not in Xtr_e.columns and "G2" not in Xtr_e.columns, "G1/G2 present in Early."

print("✅ All integrity validations passed.")

✅ All integrity validations passed.


In [114]:
# Check for nulls and non-numeric types
missing_value_summary = pd.DataFrame({
    "Dataset": ["Xtr_f", "Xte_f", "Xtr_e", "Xte_e", "ytr", "yte"],
    "Missing_Values": [Xtr_f.isna().sum().sum(), Xte_f.isna().sum().sum(),
                       Xtr_e.isna().sum().sum(), Xte_e.isna().sum().sum(),
                       ytr.isna().sum(), yte.isna().sum()]
})
display(missing_value_summary)
assert (missing_value_summary["Missing_Values"] == 0).all(), "Missing values found!"

# Numeric and Infinity check
for name, df in feature_objects.items():
    non_numeric = df.select_dtypes(exclude=["number", "bool"]).columns.tolist()
    assert len(non_numeric) == 0, f"{name} contains non-numeric columns: {non_numeric}"

    numeric_array = df.to_numpy(dtype=float)
    assert not np.isinf(numeric_array).any(), f"Infinite values found in {name}."

print("✅ Data is clean, numeric, and finite.")

,Dataset,Missing_Values
0,Xtr_f,0
1,Xte_f,0
2,Xtr_e,0
3,Xte_e,0
4,ytr,0
5,yte,0


✅ Data is clean, numeric, and finite.


In [115]:
# Standardize target names
ytr = ytr.copy(); ytr.name = "G3"
yte = yte.copy(); yte.name = "G3"

# Summary of Artifacts
artifact_shapes = pd.DataFrame({
    "Artifact": ["X_train_full", "X_test_full", "X_train_early", "X_test_early", "y_train", "y_test"],
    "Rows": [Xtr_f.shape[0], Xte_f.shape[0], Xtr_e.shape[0], Xte_e.shape[0], ytr.shape[0], yte.shape[0]],
    "Columns": [Xtr_f.shape[1], Xte_f.shape[1], Xtr_e.shape[1], Xte_e.shape[1], 1, 1]
})
display(artifact_shapes)

# Parquet Paths
parquet_paths = {
    "X_train_full": OUTPUT_DIR / "X_train_full.parquet",
    "X_test_full": OUTPUT_DIR / "X_test_full.parquet",
    "X_train_early": OUTPUT_DIR / "X_train_early.parquet",
    "X_test_early": OUTPUT_DIR / "X_test_early.parquet",
    "y_train": OUTPUT_DIR / "y_train.parquet",
    "y_test": OUTPUT_DIR / "y_test.parquet"
}

for name, path in parquet_paths.items():
    print(f"{name}: {path}")

,Artifact,Rows,Columns
0,X_train_full,316,41
1,X_test_full,79,41
2,X_train_early,316,39
3,X_test_early,79,39
4,y_train,316,1
5,y_test,79,1


X_train_full: /content/drive/MyDrive/GSSRP/data/processed/session22_splits/X_train_full.parquet
X_test_full: /content/drive/MyDrive/GSSRP/data/processed/session22_splits/X_test_full.parquet
X_train_early: /content/drive/MyDrive/GSSRP/data/processed/session22_splits/X_train_early.parquet
X_test_early: /content/drive/MyDrive/GSSRP/data/processed/session22_splits/X_test_early.parquet
y_train: /content/drive/MyDrive/GSSRP/data/processed/session22_splits/y_train.parquet
y_test: /content/drive/MyDrive/GSSRP/data/processed/session22_splits/y_test.parquet


In [116]:
# --- Fix: Define Missing Variables and Run Final Validation ---

# 1. Calculate the required variables that were missing
nonmatching_rows = (~test_index_comparison["Indices_Match"]).sum()
only_in_full_test = full_test_index_set - early_test_index_set
only_in_early_test = early_test_index_set - full_test_index_set

# 2. Run the final assertions
assert student_activity_summary["Result"].all(), (
    "One or more student activity checks failed."
)
assert nonmatching_rows == 0, (
    "At least one test-set index does not match."
)
assert len(only_in_full_test) == 0, (
    "Some test indices appear only in Xte_f."
)
assert len(only_in_early_test) == 0, (
    "Some test indices appear only in Xte_e."
)

# 3. Print the success message
print("=" * 68)
print("SESSION 22 SECTION 5 COMPLETED SUCCESSFULLY")
print("=" * 68)
print(f"Number of test observations: {len(Xte_f)}")
print(f"Identical test index order: {Xte_f.index.equals(Xte_e.index)}")
print(f"Identical test index set: {set(Xte_f.index) == set(Xte_e.index)}")
print(f"Nonmatching test rows: {nonmatching_rows}")
print(f"Test features match the shared target: {Xte_f.index.equals(yte.index) and Xte_e.index.equals(yte.index)}")
print("\nConclusion: both modeling scenarios use the same students in the test set.")

SESSION 22 SECTION 5 COMPLETED SUCCESSFULLY
Number of test observations: 79
Identical test index order: True
Identical test index set: True
Nonmatching test rows: 0
Test features match the shared target: True

Conclusion: both modeling scenarios use the same students in the test set.
